In [2]:
import numpy as np
import pandas as pd

npz_path = "/content/drive/MyDrive/symbolic_data/feynman_datapoints.npz"
csv_path = "/content/drive/MyDrive/symbolic_data/feynman_new.csv"

data = np.load(npz_path)
df = pd.read_csv(csv_path)

print("Total equations:", len(df))
print("Example prefix:", df.iloc[0]["expression_prefix_masked"])
print("Example X shape:", data["X_0"].shape)

Total equations: 100
Example prefix: DIV exp DIV POW NEG theta <C_0> <C_0> sqrt MUL <C_0> pi
Example X shape: (500, 1)


In [5]:
features = []
prefix_list = []
N=len(df)
for i in range(N):

    X = data[f"X_{i}"]      # (500, vars)
    y = data[f"y_{i}"]      # (500,)

    y = y.reshape(-1,1)

    xy = np.concatenate([X, y], axis=1)

    features.append(xy)
    prefix_list.append(df.iloc[i]["expression_prefix_masked"])

In [7]:
from collections import Counter

tokens = []

for eq in prefix_list:
    tokens.extend(eq.split())

vocab = ["<PAD>", "<BOS>", "<EOS>"] + list(set(tokens))

stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for s,i in stoi.items()}

vocab_size = len(vocab)

print("Vocab size:", vocab_size)

Vocab size: 115


In [9]:
MAX_LEN = 30

def encode(eq):

    tokens = eq.split()

    ids = [stoi["<BOS>"]]

    for t in tokens:
        ids.append(stoi[t])

    ids.append(stoi["<EOS>"])

    while len(ids) < MAX_LEN:
        ids.append(stoi["<PAD>"])

    return ids[:MAX_LEN]

targets = np.array([encode(eq) for eq in prefix_list])

In [12]:
max_vars = 0

for i in range(N):

    X = data[f"X_{i}"]

    vars_i = X.shape[1]

    max_vars = max(max_vars, vars_i)

print("Max variables:", max_vars)

Max variables: 9


In [14]:
features = []
prefix_list = []

for i in range(N):

    X = data[f"X_{i}"]      # (500, vars)
    y = data[f"y_{i}"]      # (500,)

    vars_i = X.shape[1]

    if vars_i < max_vars:

        pad = np.zeros((500, max_vars - vars_i))

        X = np.concatenate([X, pad], axis=1)

    y = y.reshape(-1,1)

    xy = np.concatenate([X, y], axis=1)

    features.append(xy)
    prefix_list.append(df.iloc[i]["expression_prefix_masked"])

features = np.array(features)

In [27]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class EquationDataset(Dataset):

    def __init__(self, data, df, stoi, max_len):

        features = []
        targets = []

        N = len(df)

        # detect max variables
        max_vars = 0
        for i in range(N):
            X = data[f"X_{i}"]
            max_vars = max(max_vars, X.shape[1])

        for i in range(N):

            X = data[f"X_{i}"]
            y = data[f"y_{i}"]

            vars_i = X.shape[1]

            if vars_i < max_vars:
                pad = np.zeros((500, max_vars-vars_i))
                X = np.concatenate([X,pad],axis=1)

            y = y.reshape(-1,1)

            xy = np.concatenate([X,y],axis=1)

            features.append(xy)

            # encode prefix
            tokens = df.iloc[i]["expression_prefix_masked"].split()

            ids = [stoi["<BOS>"]]

            for t in tokens:
                ids.append(stoi[t])

            ids.append(stoi["<EOS>"])

            while len(ids) < max_len:
                ids.append(stoi["<PAD>"])

            targets.append(ids[:max_len])

        self.features = torch.tensor(np.array(features),dtype=torch.float32)
        self.targets = torch.tensor(np.array(targets),dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
        return self.features[idx], self.targets[idx]


In [28]:
dataset = EquationDataset(data, df, stoi, MAX_LEN)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

In [29]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):

    def __init__(self,d_model,max_len=1000):

        super().__init__()

        pe = torch.zeros(max_len,d_model)

        pos = torch.arange(0,max_len).unsqueeze(1)

        div = torch.exp(
            torch.arange(0,d_model,2)*(-math.log(10000)/d_model)
        )

        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)

        self.pe = pe.unsqueeze(0)

    def forward(self,x):

        return x + self.pe[:,:x.size(1)].to(x.device)

In [35]:
class PointEmbedding(nn.Module):

    def __init__(self,input_dim,d_model):

        super().__init__()

        self.linear = nn.Linear(input_dim,d_model)

    def forward(self,x):

        return self.linear(x)

In [36]:
class PointEncoder(nn.Module):

    def __init__(self,d_model=256,nhead=8,layers=4):

        super().__init__()

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(layer,layers)

    def forward(self,x):

        return self.encoder(x)

In [37]:
class Predictor(nn.Module):

    def __init__(self,d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(d_model,d_model),

            nn.ReLU(),

            nn.Linear(d_model,d_model)
        )

    def forward(self,x):

        return self.net(x)

In [38]:
class LMJEPA(nn.Module):

    def __init__(self,input_dim,d_model=256):

        super().__init__()

        self.embed = PointEmbedding(input_dim,d_model)

        self.context_encoder = PointEncoder(d_model)

        self.target_encoder = PointEncoder(d_model)

        self.predictor = Predictor(d_model)

    def encode_context(self,x):

        x = self.embed(x)

        h = self.context_encoder(x)

        return h.mean(dim=1)

    def encode_target(self,x):

        x = self.embed(x)

        with torch.no_grad():

            h = self.target_encoder(x)

        return h.mean(dim=1)

    def forward(self,context,target):

        context_repr = self.encode_context(context)

        target_repr = self.encode_target(target)

        pred = self.predictor(context_repr)

        return pred,target_repr

In [40]:
def update_target_encoder(model,momentum=0.996):

    for p,q in zip(
        model.context_encoder.parameters(),
        model.target_encoder.parameters()
    ):

        q.data = momentum*q.data + (1-momentum)*p.data

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

device = "cuda" if torch.cuda.is_available() else "cpu"

# Instantiate the LMJEPA model
# input_dim = max_vars (9) + 1 (for y) = 10
model = LMJEPA(input_dim=max_vars + 1).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=300
)
criterion = nn.MSELoss()

for epoch in range(100):

    model.train()

    total_loss = 0

    # Unpack the batch into features (X_batch) and targets (y_target_batch)
    # and move features to the device
    for X_batch, y_target_batch in loader:

        X = X_batch.to(device) 

        perm = torch.randperm(500)

        context_idx = perm[:350]
        target_idx = perm[350:]

        context = X[:,context_idx,:]
        target = X[:,target_idx,:]

        pred,target_repr = model(context,target)

        loss = criterion(pred,target_repr)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        update_target_encoder(model)

        total_loss += loss.item()

    scheduler.step()

    print("epoch",epoch,"loss",total_loss)


epoch 0 loss 5.902375280857086
epoch 1 loss 4.929159462451935
epoch 2 loss 4.232101559638977
epoch 3 loss 3.5370879769325256
epoch 4 loss 2.860738694667816
epoch 5 loss 2.3673622012138367
epoch 6 loss 1.9533686339855194
epoch 7 loss 1.6363559812307358
epoch 8 loss 1.3885502517223358
epoch 9 loss 1.2528922110795975
epoch 10 loss 1.1257366240024567
epoch 11 loss 0.9709288626909256
epoch 12 loss 0.8298627063632011
epoch 13 loss 0.7315168082714081
epoch 14 loss 0.7301504760980606
epoch 15 loss 0.6155881285667419
epoch 16 loss 0.5744412913918495
epoch 17 loss 0.5168459564447403
epoch 18 loss 0.4883319176733494
epoch 19 loss 0.45137014240026474
epoch 20 loss 0.4237104691565037
epoch 21 loss 0.40727758035063744
epoch 22 loss 0.38234977796673775
epoch 23 loss 0.41801758110523224
epoch 24 loss 0.3178213778883219
epoch 25 loss 0.40763887017965317
epoch 26 loss 0.3140312470495701
epoch 27 loss 0.34861861914396286
epoch 28 loss 0.31742904894053936
epoch 29 loss 0.3241627812385559
epoch 30 loss 0.2

In [47]:
model.eval()

with torch.no_grad():

    X = dataset[0][0].unsqueeze(0).to(device) # Corrected line: Access the features tensor at index 0

    perm = torch.randperm(500)

    context = X[:,perm[:350],:]
    target = X[:,perm[350:],:]

    pred,target_repr = model(context,target)

    print("embedding distance:",torch.norm(pred-target_repr))

embedding distance: tensor(3.6989, device='cuda:0')


In [49]:
import torch

checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict()
}

torch.save(checkpoint, "/content/drive/MyDrive/symbolic_data/basic_jepa_encoder.pt")

print("Model saved successfully.")

Model saved successfully.


In [50]:
torch.save(
    {
        "embed": model.embed.state_dict(),
        "encoder": model.context_encoder.state_dict()
    },
    "/content/drive/MyDrive/symbolic_data/Just_jepa_encoder.pt"
)

print("Encoder saved.")

Encoder saved.


In [51]:
checkpoint = torch.load("/content/drive/MyDrive/symbolic_data/basic_jepa_encoder.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

start_epoch = checkpoint["epoch"]

print("Checkpoint loaded.")

Checkpoint loaded.


In [52]:
encoder_weights = torch.load("/content/drive/MyDrive/symbolic_data/Just_jepa_encoder.pt")

model.embed.load_state_dict(encoder_weights["embed"])

model.context_encoder.load_state_dict(encoder_weights["encoder"])

<All keys matched successfully>

In [53]:
for p in model.context_encoder.parameters():
    p.requires_grad = False

In [54]:
# decoder training
from collections import Counter

tokens = []

for eq in df["expression_prefix_masked"]:
    tokens.extend(eq.split())

vocab = ["<PAD>","<BOS>","<EOS>"] + list(set(tokens))

stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for s,i in stoi.items()}

vocab_size = len(vocab)

MAX_LEN = 30

In [56]:
def encode(eq):

    ids = [stoi["<BOS>"]]

    for t in eq.split():
        ids.append(stoi[t])

    ids.append(stoi["<EOS>"])

    while len(ids) < MAX_LEN:
        ids.append(stoi["<PAD>"])

    return ids[:MAX_LEN]

targets = torch.tensor(
    [encode(eq) for eq in df["expression_prefix_masked"]],
    dtype=torch.long
)

In [58]:
latents = []

model.eval()

with torch.no_grad():

    # Iterate through the dataset, unpacking the features and ignoring the targets
    for features_batch, _ in dataset:

        # Add a batch dimension and move the features tensor to the device
        X = features_batch.unsqueeze(0).to(device)

        # Use the correct encoding method from the model
        emb = model.encode_context(X)

        latents.append(emb.cpu())

latents = torch.cat(latents)

In [60]:
latents.shape

torch.Size([100, 256])

In [77]:
class DecoderDataset(Dataset):

    def __init__(self, latents, targets):

        self.latents = latents
        self.targets = targets

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):

        return self.latents[idx], self.targets[idx]

In [78]:
decoder_dataset = DecoderDataset(latents, targets)

decoder_loader = DataLoader(
    decoder_dataset,
    batch_size=16,
    shuffle=True
)

In [79]:
import math

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        pos = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000)/d_model)
        )

        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)

        self.pe = pe.unsqueeze(0)

    def forward(self,x):

        return x + self.pe[:,:x.size(1)].to(x.device)

In [80]:
class EquationDecoder(nn.Module):

    def __init__(self, vocab_size, d_model=256):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=8,
            dim_feedforward=1024,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(layer, num_layers=6)

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, tokens, memory, tgt_mask):

        x = self.embedding(tokens)

        x = self.pos(x)

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask
        )

        logits = self.fc(out)

        return logits

In [87]:
def token_accuracy(logits, target):

    pred = logits.argmax(-1)

    mask = (target != stoi["<PAD>"])

    correct = (pred == target) & mask

    acc = correct.sum().float() / mask.sum()

    return acc.item()

In [82]:
device = "cuda" if torch.cuda.is_available() else "cpu"

decoder = EquationDecoder(vocab_size).to(device)

optimizer = torch.optim.AdamW(
    decoder.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=400
)

criterion = nn.CrossEntropyLoss(
    ignore_index=stoi["<PAD>"]
)

In [84]:
def generate_mask(seq_len):

    mask = torch.triu(
        torch.ones(seq_len, seq_len) * float("-inf"),
        diagonal=1
    )

    return mask

In [ ]:
for epoch in range(400):

    decoder.train()

    total_loss = 0
    total_acc = 0

    for latent_batch_original, y in decoder_loader:

        latent_batch_original = latent_batch_original.to(device)
        y = y.to(device)

        inp = y[:,:-1]
        target = y[:,1:]

        seq_len = inp.shape[1]

        tgt_mask = generate_mask(seq_len).to(device)

        memory = latent_batch_original.unsqueeze(1) # This latent vector will serve as the memory for the decoder

        logits = decoder(inp, memory, tgt_mask)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            target.reshape(-1)
        )

        acc = token_accuracy(logits, target)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_acc += acc

    scheduler.step()

    print(
        "epoch",
        epoch,
        "loss",
        total_loss/len(decoder_loader),
        "acc",
        total_acc/len(decoder_loader)
    )

epoch 0 loss 4.513627767562866 acc 0.08949658593961171
epoch 1 loss 3.9410174914768765 acc 0.1943933985063008
epoch 2 loss 3.6975515569959367 acc 0.19208790361881256
epoch 3 loss 3.592413834163121 acc 0.1971470479454313
epoch 4 loss 3.484330415725708 acc 0.203719362616539
epoch 5 loss 3.4544107913970947 acc 0.19044771577630723
epoch 6 loss 3.3637567247663225 acc 0.21721182763576508
epoch 7 loss 3.3071249212537492 acc 0.22364355197974614
epoch 8 loss 3.26130941935948 acc 0.23532543863568986
epoch 9 loss 3.1915478706359863 acc 0.2608079420668738
epoch 10 loss 3.11190812928336 acc 0.2800146873508181
epoch 11 loss 3.098513807569231 acc 0.27329582827431814
epoch 12 loss 3.0527613503592357 acc 0.28651479738099234
epoch 13 loss 3.0086563314710344 acc 0.3105891602379935
epoch 14 loss 2.924445254462106 acc 0.3138934629304068
epoch 15 loss 2.8703610556466237 acc 0.3348314208643777
epoch 16 loss 2.796992097582136 acc 0.3212764986923763
epoch 17 loss 2.7239560059138705 acc 0.3699875090803419
epoch

In [90]:
decoder.eval()

with torch.no_grad():

    # The latents[0] is (d_model,). unsqueeze(0) makes it (1, d_model).
    # We need to unsqueeze again to get (1, 1, d_model) for the sequence length dimension.
    memory = latents[0].unsqueeze(0).unsqueeze(1).to(device)

    y = targets[0].unsqueeze(0).to(device)

    logits = decoder(y[:,:-1], memory, generate_mask(MAX_LEN-1).to(device))

    pred = logits.argmax(-1)

print("true:", targets[0])
print("pred:", pred[0].cpu())

true: tensor([  1,  61,  25,  61,  45,  28, 106,  70,  70,  66,  56,  70,  19,   2,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0])
pred: tensor([ 61,  25,  61,  45,  28, 106,  70,  70,  66,  56,  70,  19,   2,   2,
          2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2])


In [91]:
#test
def decode(ids):

    tokens = []

    for i in ids:

        t = itos[int(i)]

        if t == "<EOS>":
            break

        if t not in ["<BOS>", "<PAD>"]:
            tokens.append(t)

    return " ".join(tokens)

In [92]:
def generate_mask(seq_len):

    mask = torch.triu(
        torch.ones(seq_len, seq_len) * float("-inf"),
        diagonal=1
    )

    return mask

In [93]:
def generate_equation(decoder, memory, max_len=30):

    decoder.eval()

    tokens = torch.tensor([[stoi["<BOS>"]]]).to(device)

    for _ in range(max_len):

        seq_len = tokens.shape[1]

        tgt_mask = generate_mask(seq_len).to(device)

        logits = decoder(tokens, memory, tgt_mask)

        next_token = logits[:, -1].argmax(-1)

        tokens = torch.cat(
            [tokens, next_token.unsqueeze(1)],
            dim=1
        )

        if next_token.item() == stoi["<EOS>"]:
            break

    return tokens[0]

In [94]:
def encode_points(model, X):

    with torch.no_grad():

        X = X.unsqueeze(0).to(device)

        x = model.embed(X)

        tokens = model.context_encoder(x)

    return tokens

In [95]:
pool = torch.nn.AdaptiveAvgPool1d(64)

def pool_tokens(tokens):

    tokens = tokens.transpose(1,2)

    tokens = pool(tokens)

    tokens = tokens.transpose(1,2)

    return tokens

In [96]:
def predict_equation(model, decoder, X):

    tokens = encode_points(model, X)

    memory = pool_tokens(tokens)

    pred_ids = generate_equation(decoder, memory)

    return decode(pred_ids)

In [98]:
correct = 0
total = 0

model.eval()
decoder.eval()

for i in range(len(dataset)):

    X = dataset.features[i]

    true_eq = df.iloc[i]["expression_prefix_masked"]

    pred_eq = predict_equation(model, decoder, X)

    if pred_eq == true_eq:
        correct += 1

    total += 1

    print("True:", true_eq)
    print("Pred:", pred_eq)
    print()

True: DIV exp DIV POW NEG theta <C_0> <C_0> sqrt MUL <C_0> pi
Pred: DIV exp DIV POW NEG theta <C_0> <C_0> sqrt MUL <C_0> pi

True: DIV exp DIV POW NEG DIV theta sigma <C_0> <C_0> MUL sqrt MUL <C_0> pi sigma
Pred: DIV exp DIV POW NEG DIV theta sigma <C_0> <C_0> MUL sqrt MUL <C_0> pi sigma

True: DIV exp DIV POW NEG DIV SUB theta theta1 sigma <C_0> <C_0> MUL sqrt MUL <C_0> pi sigma
Pred: DIV MUL q h MUL MUL <C_0> pi m

True: sqrt ADD POW SUB x2 x1 <C_0> POW SUB y2 y1 <C_0>
Pred: DIV MUL MUL DIV <C_0> SUB gamma <C_0> kb v A

True: DIV MUL MUL G m1 m2 ADD ADD POW SUB x2 x1 <C_0> POW SUB y2 y1 <C_0> POW SUB z2 z1 <C_0>
Pred: DIV MUL MUL G m1 m2 ADD ADD POW SUB x2 x1 <C_0> POW SUB y2 y1 <C_0> POW SUB z2 z1 <C_0>

True: DIV m_0 sqrt SUB <C_0> DIV POW v <C_1> POW c <C_1>
Pred: DIV m_0 sqrt SUB <C_0> DIV POW v <C_1> POW c <C_1>

True: ADD ADD MUL x1 y1 MUL x2 y2 MUL x3 y3
Pred: MUL MUL DIV <C_0> <C_1> pr V

True: MUL mu Nn
Pred: MUL mu Nn

True: DIV MUL MUL q1 q2 r MUL MUL MUL <C_0> pi epsilon 

In [99]:
accuracy = correct / total

print("Exact Match Accuracy:", accuracy)

Exact Match Accuracy: 0.49
